# Урок 17 - Създаване на локални AI агенти с Foundry Local и Qwen

В този тетрадка изграждате **локален инженеринг асистент**, който работи изцяло на вашата работна станция. Никога не се използва облачна инференция. Асистентът може:

1. **Да вика инструменти** чрез Qwen функция, извиквана през Foundry Local.
2. **Да изброява и чете файлове** в ограничена директория на проекта.
3. **Да анализира код** с прости метрики.
4. **Да търси документация** с локален RAG (Chroma).
5. **Да използва локален MCP сървър** (игнорира се плавно, ако няма конфигурация).

Кодът на агента изглежда почти идентично с уроците за облака — единствено крайният клиент се мести от облака към `localhost`.


## Настройка

Преди да стартирате тази тетрадка:

1. **Инсталирайте Microsoft Foundry Local** (вижте [документацията](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) за вашата операционна система).
2. **Изтеглете и стартирайте модел Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Инсталирайте Python пакетите по-долу.

Всичко работи локално. Машина с ~8 GB RAM е реалистичният минимум.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Свържете се с Foundry Local

`FoundryLocalManager` изтегля модела при необходимост, стартира локалната услуга и ни предоставя **съвместим с OpenAI крайна точка**. След това насочваме стандартния OpenAI SDK към нея. API ключът е локален заместител — не се използват облачни идентификационни данни.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Локални инструменти (пясъчник за операции с файлове)

Създаваме малък примерен проект на диска и след това дефинираме инструменти, обхванати в корена на този проект. Проверката на пясъчника е важна дори локално: инструмент, който чете произволни пътища, работи с правата на вашия потребител и може да осъществи достъп до всичко, до което и вие можете.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Локален RAG с Chroma

Вграждаме малък набор от документирани откъси в локална колекция Chroma. Chroma работи в процеса и съхранява вектори на диск — без сървър, без облак. Инструментът `search_docs` извлича най-релевантните откъси за дадена заявка.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Цикълът за извикване на инструменти

Сега регистрираме инструментите с модела, използвайки схемата за инструменти на OpenAI, и стартираме стандартния цикъл за извикване на инструменти — моделът изисква инструмент, ние го изпълняваме локално, подаваме резултата обратно и повтаряме, докато моделът не произведе окончателен отговор. Надеждното извикване на функции на Qwen е това, което прави това възможно на устройството.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Локален MCP (по избор)

MCP е транспорт, а не облачна услуга — MCP сървър може да работи като локален процес чрез `stdio`. Клетката по-долу показва как да се свържете с локален MCP сървър, ако имате такъв конфигуриран (например файлов сървър). Тя пропуска внимателно, когато `LOCAL_MCP_COMMAND` не е зададен, така че тетрадката все пак се изпълнява изцяло без него.

Забележка за сигурността: локален MCP сървър работи с разрешенията на вашия потребител. Ограничавайте го до проектна директория и валидирайте изходните му данни преди да действате според тях.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Резюме

Изградихте инженерски асистент, който работи изцяло на вашата машина:

- **Foundry Local** поддържаше модел **Qwen** зад OpenAI-съвместим интерфейс — така кодът на агента съответства на уроците от облака.
- **Изолирани инструменти** предоставиха на агента достъп до файлове и анализ на код без да напуска директорията на проекта.
- **Chroma** осигури **локален RAG** върху документацията.
- **Local MCP** показа как да се използва екосистемата на MCP офлайн.

Никога не беше използван облачен извод.

### Предизвикателство

Разширете локалния агент да:

1. **Работи с множество MCP сървъри** — свържете файлов сървър и git сървър и позволете на агента да избира между тях.
2. **Използва локална памет** — запазва кратка история на разговора на диск, за да помни асистентът по-ранни реплики след рестартиране на бележника.
3. **Поддържа локална многоагентна оркестрация** — добавете втори локален агент (например прегледвач) и нека двамата да си сътрудничат по задача.

В следващия урок ще научите как да защитите внедрените AI агенти.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
